In [17]:
import pandas as pd

graph_data = pd.read_csv("../..//dataset/graph-format-data/noise_label_data/train_correct_df.csv")
import ast
def parse_and_strip_benign(label_str):
    label_set = ast.literal_eval(label_str)  # 安全地解析字符串为集合
    new_set = label_set - {'BENIGN'}

    return str(new_set) if new_set else str(label_set)

# 应用函数处理每一项
graph_data['cleaned_group_label'] = graph_data['group_label'].apply(parse_and_strip_benign)
# 自定义映射字典（字符串需完全匹配）
label_map = {
    "{'BENIGN'}": 0,
    "{'Bot'}": 1,
    "{'PortScan'}": 2,
    "{'DDoS'}": 3,
    "{'SSH-Patator'}": 4,
    "{'DoS Hulk'}": 5,
    "{'Web Attack ?Brute Force'}": 6,
    "{'Web Attack ?XSS'}": 7,
    "{'DoS slowloris'}": 8,
    "{'Infiltration'}": 9,
    "{'DoS Slowhttptest'}": 10,
    "{'DoS GoldenEye'}": 11,
    "{'FTP-Patator'}": 12,
    "{'Web Attack ?Sql Injection'}": 13
}
# 获得id_label的map
# 映射到新的列
graph_data['label_id'] = graph_data['cleaned_group_label'].map(label_map)
# 查看映射结果（可选）
print(graph_data[['cleaned_group_label', 'label_id']].value_counts())

cleaned_group_label            label_id
{'BENIGN'}                     0           8849
{'Bot'}                        1             24
{'DDoS'}                       3              9
{'PortScan'}                   2              9
{'SSH-Patator'}                4              7
{'DoS Hulk'}                   5              6
{'Web Attack ?Brute Force'}    6              6
{'Infiltration'}               9              5
{'Web Attack ?XSS'}            7              5
{'DoS slowloris'}              8              5
{'DoS Slowhttptest'}           10             4
{'DoS GoldenEye'}              11             3
{'FTP-Patator'}                12             2
{'Web Attack ?Sql Injection'}  13             2
dtype: int64


In [18]:
# 随机污染标签
# 0改成1~13任意标签；1~13的标签一半概率改为0，一般概率改为其他标签
import random
#random.seed(22)

def noise_label(train_df,ratio):
    # 随机选择要污染的行
    rows_to_pollute = train_df.sample(frac=ratio)
    # 对污染的行遍历
    for index, row in rows_to_pollute.iterrows():
        # 获得标签
        label = row['label_id']
        # 根据标签进行污染
        if label == 0:
            # 0改成1~13任意标签
            new_label = random.randint(1, 13)
        else:
            # 1~13的标签一半概率改为0，一半概率改为其他标签
            if random.random() <= 0.5:
                train_df.loc[index, 'label_id'] = 0
            else:
                new_label = random.randint(1, 13)
        train_df.loc[index, 'label_id'] = new_label
        #print('污染的行：',index,'标签：',row['label_id'],'污染后标签：',new_label)
    return train_df

In [19]:
import math

def proportional_sample(df, label_col, ratio):
    """
    对每个标签进行等比例采样，最少保留一个样本。
    
    参数:
    - df: 原始 DataFrame
    - label_col: 分组标签列名
    - ratio: 采样比例，如 0.01 表示 1%
    
    返回:
    - 采样后的 DataFrame
    """
    sampled_dfs = []
    
    for label_value, group_df in df.groupby(label_col):
        sample_size = max(1, math.floor(len(group_df) * ratio))
        sampled = group_df.sample(n=sample_size, random_state=42)  # 设置随机种子确保可复现
        sampled_dfs.append(sampled)
    
    return pd.concat(sampled_dfs).reset_index(drop=True)

In [20]:
def get_weights(graph_data):
    """ 获取代价敏感损失函数的权重
    Args:
        graph_data: 图数据
    Returns:
        权重列表
    """
    # 计算cleaned_group_label的比例，来确定代价敏感损失函数的权重
    # 统计频数
    counts = graph_data['label_id'].value_counts().to_dict()
    # 总样本数
    total = sum(counts.values())
    # 对counts按照key排序
    counts = dict(sorted(counts.items()))
    # 5.按类别顺序计算权重
    weights = [0 for _ in range(len(id_to_label))]
    for c in counts.keys():
        count = counts.get(c, 0)  # 避免除以 0
        if count == 0:
            continue
        now_weight = round(total / count,2)

        weights[int(c)] = now_weight
    # 6. 输出
    print("类别权重对应顺序：")
    print(counts.keys())
    print("对应权重列表：")
    print([round(w, 2) for w in weights])
    return weights

In [21]:
noise_rate_list = [0.01,0.05,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
weights_dic = {}
ver = 5
for noise_rate in noise_rate_list:
    train_df = graph_data.copy()
    noise_train_df = noise_label(train_df, noise_rate)
    # 反映射
    id_to_label = {v: k for k, v in label_map.items()}
    noise_train_df['now_cleaned_group_label'] = noise_train_df['label_id'].map(id_to_label)
    # 采样10%作为微调样本
    sample_ratio = 0.1
    sampled_df = proportional_sample(noise_train_df, 'now_cleaned_group_label', ratio=sample_ratio)
    weights_dic[noise_rate] = get_weights(sampled_df)
    output_path = f'../..//dataset/graph-format-data/noise-lossWeights_label_data{ver}/train_sample/noise_{noise_rate}_sample_{sample_ratio}.csv'
    sampled_df.to_csv(output_path, index=False)

类别权重对应顺序：
dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])
对应权重列表：
[1.02, 297.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0]
类别权重对应顺序：
dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])
对应权重列表：
[1.05, 177.2, 221.5, 295.33, 221.5, 221.5, 295.33, 295.33, 295.33, 221.5, 295.33, 295.33, 221.5, 295.33]
类别权重对应顺序：
dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])
对应权重列表：
[1.11, 98.44, 147.67, 147.67, 110.75, 126.57, 126.57, 147.67, 126.57, 126.57, 147.67, 177.2, 98.44, 126.57]
类别权重对应顺序：
dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])
对应权重列表：
[1.26, 63.43, 68.31, 68.31, 63.43, 63.43, 68.31, 68.31, 55.5, 59.2, 52.24, 68.31, 74.0, 63.43]
类别权重对应顺序：
dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])
对应权重列表：
[1.43, 40.27, 44.3, 42.19, 40.27, 42.19, 40.27, 46.63, 44.3, 40.27, 42.19, 46.63, 46.63, 46.63]
类别权重对应顺序：
dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])
对应权重列表：
[1.68, 31.75, 32.93, 32.93, 29.63, 31.75, 

In [22]:
print(weights_dic)

{0.01: [1.02, 297.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0, 891.0], 0.05: [1.05, 177.2, 221.5, 295.33, 221.5, 221.5, 295.33, 295.33, 295.33, 221.5, 295.33, 295.33, 221.5, 295.33], 0.1: [1.11, 98.44, 147.67, 147.67, 110.75, 126.57, 126.57, 147.67, 126.57, 126.57, 147.67, 177.2, 98.44, 126.57], 0.2: [1.26, 63.43, 68.31, 68.31, 63.43, 63.43, 68.31, 68.31, 55.5, 59.2, 52.24, 68.31, 74.0, 63.43], 0.3: [1.43, 40.27, 44.3, 42.19, 40.27, 42.19, 40.27, 46.63, 44.3, 40.27, 42.19, 46.63, 46.63, 46.63], 0.4: [1.68, 31.75, 32.93, 32.93, 29.63, 31.75, 32.93, 31.75, 30.66, 35.56, 31.75, 35.56, 31.75, 30.66], 0.5: [2.01, 26.09, 26.88, 24.64, 24.64, 23.34, 26.09, 26.09, 26.88, 27.72, 27.72, 25.34, 26.88, 25.34], 0.6: [2.5, 20.14, 21.61, 22.72, 18.08, 21.61, 22.72, 25.31, 23.32, 22.72, 19.26, 22.72, 22.15, 21.1], 0.7: [3.34, 18.52, 18.91, 18.14, 18.91, 18.52, 20.2, 18.14, 17.1, 18.91, 17.78, 19.76, 18.14, 18.52], 0.8: [5.03, 15.54, 15.54, 16.11, 17.72, 17.04, 15.28,

# 更改训练集对应label

In [23]:
import pandas as pd

import ast
def parse_and_strip_benign(label_str):
    label_set = ast.literal_eval(label_str)  # 安全地解析字符串为集合
    new_set = label_set - {'BENIGN'}

    return str(new_set) if new_set else str(label_set)

train_df_now = pd.read_csv('../..//dataset/graph-format-data/noise_label_data/train_correct_df.csv')
graph_data = train_df_now
# 应用函数处理每一项
graph_data['cleaned_group_label'] = graph_data['group_label'].apply(parse_and_strip_benign)
# 自定义映射字典（字符串需完全匹配）
label_map = {
    "{'BENIGN'}": 0,
    "{'Bot'}": 1,
    "{'PortScan'}": 2,
    "{'DDoS'}": 3,
    "{'SSH-Patator'}": 4,
    "{'DoS Hulk'}": 5,
    "{'Web Attack ?Brute Force'}": 6,
    "{'Web Attack ?XSS'}": 7,
    "{'DoS slowloris'}": 8,
    "{'Infiltration'}": 9,
    "{'DoS Slowhttptest'}": 10,
    "{'DoS GoldenEye'}": 11,
    "{'FTP-Patator'}": 12,
    "{'Web Attack ?Sql Injection'}": 13
}
# 获得id_label的map
# 映射到新的列
ver = 5
graph_data['label_id'] = graph_data['cleaned_group_label'].map(label_map)
noise_rate_list = [0.01,0.05,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
for noise_rate in noise_rate_list:
    noise_label_path = f'../..//dataset/graph-format-data/noise-lossWeights_label_data{ver}/train_sample/noise_{noise_rate}_sample_0.1.csv'
    noise_label_df = pd.read_csv(noise_label_path)
    # 按照id进行左连接
    # 重命名label_id列名
    graph_data.rename(columns={'label_id': 'label_id_origin'}, inplace=True)
    noise_label_df.rename(columns={'label_id': 'label_id_noise'}, inplace=True)
    train_new_df = pd.merge(graph_data, noise_label_df[['id', 'label_id_noise']], on='id', how='left')
    train_new_df.reset_index(drop=True, inplace=True)
    # 当label_id_y不为空时,label_id=label_id_y
    # 当label_id_y为空时,label_id=label_id_x
    train_new_df['label_id'] = train_new_df['label_id_noise'].fillna(train_new_df['label_id_origin'])
    train_new_df['label_id'] = train_new_df['label_id'].astype(int)
    output_path = f'../..//dataset/graph-format-data/noise-lossWeights_label_data{ver}/train_graph/train_noise_{noise_rate}.csv'
    train_new_df.to_csv(output_path, index=False)